In [1]:
# Install required packages
!pip install opencv-python
!pip install easyocr
!pip install matplotlib
!pip install pillow
!pip install ipywidgets
!pip install pytesseract
!pip install opencv-contrib-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [2]:
# Core libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import json
from PIL import Image
import logging
import io
import base64
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure logging and create directories
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.makedirs('uploaded_documents', exist_ok=True)
os.makedirs('processed_results', exist_ok=True)

# For UI components
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import FileUpload, Button, Output, Label, VBox, HBox, HTML
# Initialize EasyOCR with reduced memory usage (one language at a time)
def initialize_easyocr():
    try:
        # First try with just English to avoid memory issues
        print("Initializing EasyOCR with English only...")
        reader = easyocr.Reader(['en'], gpu=False)
        print("✅ EasyOCR initialized successfully with English")

        # If that succeeds, try adding more languages one by one
        try:
            print("Adding French language model...")
            reader = easyocr.Reader(['en', 'fr'], gpu=False)
            print("✅ French language model added successfully")

            try:
                print("Adding Arabic language model...")
                reader = easyocr.Reader(['en', 'fr', 'ar'], gpu=False)
                print("✅ Arabic language model added successfully")
                return reader
            except Exception as e:
                print(f"⚠️ Could not load Arabic model: {str(e)}")
                print("Continuing with English and French only")
                return reader
        except Exception as e:
            print(f"⚠️ Could not load French model: {str(e)}")
            print("Continuing with English only")
            return reader
    except Exception as e:
        print(f"❌ Error initializing EasyOCR: {str(e)}")
        print("Please check your installation and try restarting the kernel")
        return None
    # Import OCR after all other imports to isolate any issues
print("Importing EasyOCR...")
import easyocr

# Initialize with memory-saving approach
print("Starting EasyOCR initialization...")
reader = initialize_easyocr()

Importing EasyOCR...


Starting EasyOCR initialization...
Initializing EasyOCR with English only...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

✅ EasyOCR initialized successfully with English
Adding French language model...


Progress: |██████████████████████████████████████████████████| 100.0% Complete

✅ French language model added successfully
Adding Arabic language model...
⚠️ Could not load Arabic model: Arabic is only compatible with English, try lang_list=["ar","fa","ur","ug","en"]
Continuing with English and French only


In [13]:
# Core libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import json
from PIL import Image
import logging
import io
import base64
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
# Add these imports at the top
try:
    from skimage.metrics import structural_similarity as ssim
except ImportError:
    try:
        # Install if missing
        import pip
        pip.main(['install', 'scikit-image'])
        from skimage.metrics import structural_similarity as ssim
    except Exception:
        # Define fallback if installation fails
        def ssim(img1, img2):
            return 0.5  # Default fallback value

# Configure logging and create directories
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.makedirs('uploaded_documents', exist_ok=True)
os.makedirs('processed_results', exist_ok=True)

# For UI components
from IPython.display import display, HTML, Javascript
import ipywidgets as widgets
from ipywidgets import FileUpload, Button, Output, Label, VBox, HBox

# Import OCR after all other imports
import easyocr
import uuid

# Initialize EasyOCR with reduced memory usage
def initialize_easyocr():
    try:
        # First try with just English to avoid memory issues
        print("Initializing EasyOCR with English only...")
        reader = easyocr.Reader(['en'], gpu=False)
        print("✅ EasyOCR initialized successfully with English")

        try:
            print("Adding French language model...")
            reader = easyocr.Reader(['en', 'fr'], gpu=False)
            print("✅ French language model added successfully")

            try:
                print("Adding Arabic language model...")
                reader = easyocr.Reader(['en', 'ar'], gpu=False)
                print("✅ Arabic language model added successfully")
                return reader
            except Exception as e:
                print(f"⚠️ Could not load Arabic model: {str(e)}")
                print("Continuing with English and French only")
                return reader
        except Exception as e:
            print(f"⚠️ Could not load French model: {str(e)}")
            print("Continuing with English only")
            return reader
    except Exception as e:
        print(f"❌ Error initializing EasyOCR: {str(e)}")
        print("Please check your installation and try restarting the kernel")
        return None

# Initialize reader
print("Starting EasyOCR initialization...")
reader = initialize_easyocr()

def save_uploaded_file(upload_widget):
    """Save an uploaded file from a widget to disk"""
    if not upload_widget.value:
        return None

    try:
        # Get the first uploaded file
        uploaded_file = list(upload_widget.value.values())[0]
        content = uploaded_file['content']

        # Try to get the filename, or generate one if not available
        if 'name' in uploaded_file:
            filename = uploaded_file['name']
        elif 'filename' in uploaded_file:
            filename = uploaded_file['filename']
        else:
            # Generate a random filename with timestamp
            ext = '.jpg'  # Default extension for images
            filename = f"cin_{uuid.uuid4().hex}_{datetime.now().strftime('%Y%m%d%H%M%S')}{ext}"

        # Save to disk
        save_path = os.path.join('uploaded_documents', filename)
        with open(save_path, 'wb') as f:
            f.write(content)

        logger.info(f"Saved uploaded file: {save_path}")
        return save_path

    except Exception as e:
        logger.error(f"Error saving uploaded file: {str(e)}")
        return None

def save_base64_image(base64_data, prefix="webcam"):
    """Save a base64-encoded image to disk"""
    try:
        if "base64," in base64_data:
            # Remove data URL prefix if present
            base64_data = base64_data.split("base64,")[1]

        # Decode base64 data
        image_data = base64.b64decode(base64_data)

        # Generate a filename with timestamp
        filename = f"{prefix}_{uuid.uuid4().hex}_{datetime.now().strftime('%Y%m%d%H%M%S')}.jpg"

        # Save to disk
        save_path = os.path.join('uploaded_documents', filename)
        with open(save_path, 'wb') as f:
            f.write(image_data)

        logger.info(f"Saved base64 image: {save_path}")
        return save_path
    except Exception as e:
        logger.error(f"Error saving base64 image: {str(e)}")
        return None

def load_image(image_path):
    """Load an image from the specified path"""
    try:
        img = cv2.imread(image_path)
        if img is None:
            raise Exception(f"Failed to load image from {image_path}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img_rgb
    except Exception as e:
        logger.error(f"Error loading image: {str(e)}")
        return None

def preprocess_image(image, resize_dim=(800, 600)):
    """Preprocess the image for better OCR and feature extraction"""
    try:
        if image is None:
            return None, None, None

        # Make a copy to avoid modifying the original
        img = image.copy()

        # Resize
        img = cv2.resize(img, resize_dim)

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # Apply Gaussian blur to reduce noise
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        # Apply adaptive thresholding
        thresh = cv2.adaptiveThreshold(
            blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 11, 2
        )

        return img, gray, thresh
    except Exception as e:
        logger.error(f"Error preprocessing image: {str(e)}")
        return None, None, None

def display_image(image, title="Image"):
    """Display an image with matplotlib"""
    if image is None:
        print("No image to display")
        return
    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

def extract_text_with_easyocr(image):
    """
    Extract text from image using EasyOCR with improved Arabic support
    Args:
        image: Input image

    Returns:
        List of extracted text with bounding boxes
    """
    try:
        if image is None:
            return []

        # Use the global reader
        global reader
        if reader is None:
            # Prioritize Arabic for CIN documents
            reader = easyocr.Reader(['ar', 'en', 'fr'], gpu=False)

        # Enhance image for Arabic text recognition
        # Increase contrast slightly for better Arabic character detection
        enhanced = image.copy()
        if len(enhanced.shape) == 3:
            enhanced = cv2.convertScaleAbs(enhanced, alpha=1.2, beta=10)

        # Read with Arabic-specific detail level
        results = reader.readtext(enhanced, detail=1, paragraph=False,
                                 contrast_ths=0.2, adjust_contrast=0.5)
        return results
    except Exception as e:
        logger.error(f"Error in OCR processing: {str(e)}")
        return []

def visualize_ocr_results(image, ocr_results):
    """
    Visualize OCR results by drawing bounding boxes on the image
    Args:
        image: Original image
        ocr_results: Results from EasyOCR
    """
    if image is None or not ocr_results:
        return None

    output = image.copy()

    for (bbox, text, prob) in ocr_results:
        # Convert bbox to integers
        (tl, tr, br, bl) = bbox
        tl = (int(tl[0]), int(tl[1]))
        tr = (int(tr[0]), int(tr[1]))
        br = (int(br[0]), int(br[1]))
        bl = (int(bl[0]), int(bl[1]))

        # Draw the bounding box
        cv2.polylines(output, [np.array([tl, tr, br, bl])], True, (0, 255, 0), 2)

        # Draw the text
        cv2.putText(output, text, (tl[0], tl[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    return output

def extract_name_from_cin(ocr_results):
    """
    Extract name field from CIN OCR results with improved Arabic support
    Args:
        ocr_results: Results from EasyOCR

    Returns:
        Extracted name or None if not found
    """
    try:
        # Convert results to text and position
        texts = [text for _, text, _ in ocr_results]
        full_text = " ".join(texts)

        # Arabic-specific name indicators for Tunisian CIN
        name_indicators = ["name", "nom", "الاسم", "اسم", "اللقب", "nom et prénom", "الاسم واللقب"]

        # Search for name indicators in the OCR text
        for indicator in name_indicators:
            for i, (bbox, text, _) in enumerate(ocr_results):
                # Check for partial matches for Arabic text
                if indicator in text.lower() or any(ind in text.lower() for ind in name_indicators):
                    # Name is often in the next field after the indicator
                    # or sometimes in the same field after the indicator
                    if i + 1 < len(ocr_results):
                        next_text = ocr_results[i + 1][1]
                        # Check if next text is likely a name (not a number or very short)
                        if len(next_text) > 2 and not next_text.isdigit():
                            return next_text

                    # Extract name from the same field by taking text after the indicator
                    colon_split = text.split(':')
                    if len(colon_split) > 1:
                        return colon_split[1].strip()

        # If still not found, try to find Arabic text blocks
        # Names are typically longer text blocks with Arabic characters
        for bbox, text, _ in ocr_results:
            # Check if text contains Arabic characters
            if any('\u0600' <= c <= '\u06FF' for c in text) and len(text) > 3:
                if not any(c.isdigit() for c in text):  # Not likely to be an ID number
                    return text

        # Specific fallback for Arabic names: Look for longest Arabic text
        arabic_texts = []
        for bbox, text, _ in ocr_results:
            if any('\u0600' <= c <= '\u06FF' for c in text):
                arabic_texts.append(text)

        if arabic_texts:
            # Return the longest Arabic text that's not just a single word
            candidates = [t for t in arabic_texts if len(t.split()) >= 1]
            if candidates:
                return max(candidates, key=len)

        return None
    except Exception as e:
        logger.error(f"Error extracting name: {str(e)}")
        return None

def detect_faces_in_document(image):
    """
    Detect faces in the image using OpenCV's Haar cascade classifier
    Args:
        image: Input image

    Returns:
        List of detected faces as (x, y, w, h) rectangles
    """
    if image is None:
        return []

    try:
        # Load face detector
        face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

        # Convert to grayscale for face detection
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image

        # Detect faces
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        return faces
    except Exception as e:
        logger.error(f"Error detecting faces: {str(e)}")
        return []

def extract_face_from_document(image, faces):
    """
    Extract the largest face from a document
    Args:
        image: Input image
        faces: List of face rectangles

    Returns:
        Extracted face image or None if no face is found
    """
    if image is None or len(faces) == 0:
        return None

    try:
        # Find the largest face by area
        largest_face = max(faces, key=lambda rect: rect[2] * rect[3])
        x, y, w, h = largest_face

        # Extract face region
        face_img = image[y:y+h, x:x+w]

        # Resize to standard size
        face_img = cv2.resize(face_img, (160, 160))

        return face_img
    except Exception as e:
        logger.error(f"Error extracting face: {str(e)}")
        return None

def visualize_faces(image, faces):
    """
    Draw rectangles around detected faces
    Args:
        image: Original image
        faces: List of face rectangles

    Returns:
        Image with face rectangles drawn
    """
    if image is None or len(faces) == 0:
        return image

    output = image.copy()

    for (x, y, w, h) in faces:
        cv2.rectangle(output, (x, y), (x+w, y+h), (0, 255, 0), 2)

    return output

def compare_faces(face1_path, face2_path):
    """
    Compare two face images using improved feature matching for ID documents
    Args:
        face1_path: Path to first face image (CIN document)
        face2_path: Path to second face image (webcam)

    Returns:
        Dictionary containing match result and similarity score
    """
    try:
        # Load images
        face1 = cv2.imread(face1_path)
        face2 = cv2.imread(face2_path)

        if face1 is None or face2 is None:
            raise Exception("Failed to load face images")

        # Enhanced preprocessing for ID photos
        # Convert to grayscale
        gray1 = cv2.cvtColor(face1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(face2, cv2.COLOR_BGR2GRAY)

        # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
        # This helps normalize lighting differences between ID and webcam photos
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray1_clahe = clahe.apply(gray1)
        gray2_clahe = clahe.apply(gray2)

        # Resize to same size
        height = 200
        width = 200
        gray1_resized = cv2.resize(gray1_clahe, (width, height))
        gray2_resized = cv2.resize(gray2_clahe, (width, height))

        # Apply Gaussian blur to reduce noise and small differences
        gray1_blur = cv2.GaussianBlur(gray1_resized, (5, 5), 0)
        gray2_blur = cv2.GaussianBlur(gray2_resized, (5, 5), 0)

        # Try different methods to compare faces

        # Method 1: Using SIFT feature detector (more robust than ORB)
        try:
            # Create SIFT detector
            sift = cv2.SIFT_create(nfeatures=100)  # Limit features for efficiency

            # Find keypoints and descriptors
            kp1, desc1 = sift.detectAndCompute(gray1_blur, None)
            kp2, desc2 = sift.detectAndCompute(gray2_blur, None)

            # Check if enough features were found
            if desc1 is not None and desc2 is not None and len(kp1) > 5 and len(kp2) > 5:
                # Use FLANN matcher instead of BF matcher
                FLANN_INDEX_KDTREE = 1
                index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
                search_params = dict(checks=50)
                flann = cv2.FlannBasedMatcher(index_params, search_params)

                # Find matches
                matches = flann.knnMatch(desc1, desc2, k=2)

                # Apply Lowe's ratio test
                good_matches = []
                for m, n in matches:
                    if m.distance < 0.75 * n.distance:
                        good_matches.append(m)

                similarity_sift = len(good_matches) / max(10, min(len(kp1), len(kp2)))

                # Normalize and boost slightly (since ID photos are harder to match)
                similarity_sift = min(1.0, similarity_sift * 1.5)
            else:
                similarity_sift = 0.2  # Base similarity if not enough features
        except Exception as e:
            print(f"SIFT matching failed: {e}")
            similarity_sift = 0.2  # Default value if SIFT fails

        # Method 2: Using histogram comparison (robust to lighting changes)
        try:
            # Calculate histograms
            hist1 = cv2.calcHist([gray1_blur], [0], None, [64], [0, 256])
            hist2 = cv2.calcHist([gray2_blur], [0], None, [64], [0, 256])

            # Normalize histograms
            cv2.normalize(hist1, hist1, 0, 1, cv2.NORM_MINMAX)
            cv2.normalize(hist2, hist2, 0, 1, cv2.NORM_MINMAX)

            # Compare histograms using correlation method
            similarity_hist = cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL)

            # Boost histogram similarity for ID photos
            similarity_hist = min(1.0, similarity_hist * 1.3)
        except Exception as e:
            print(f"Histogram comparison failed: {e}")
            similarity_hist = 0.3

        # Method 3: Template matching for general structure similarity
        try:
            # Use normalized cross-correlation
            result = cv2.matchTemplate(gray1_blur, gray2_blur, cv2.TM_CCOEFF_NORMED)
            similarity_template = float(np.max(result))
        except Exception as e:
            print(f"Template matching failed: {e}")
            similarity_template = 0.3

        # Method 4: Structural Similarity Index (SSIM)
        try:
            from skimage.metrics import structural_similarity as ssim
            similarity_ssim = ssim(gray1_blur, gray2_blur)
        except Exception as e:
            print(f"SSIM calculation failed: {e}")
            similarity_ssim = 0.3

        # Combine similarity scores with weights optimized for ID document comparison
        similarity = 0.35 * similarity_sift + 0.25 * similarity_hist + 0.15 * similarity_template + 0.25 * similarity_ssim

        # For ID photos, we need to be more lenient
        # Lower threshold from 0.65 to 0.45 for ID document comparison
        is_match = similarity > 0.45

        # Create result dictionary
        result = {
            "match": is_match,
            "similarity": float(similarity),
            "details": {
                "sift_score": float(similarity_sift),
                "histogram_score": float(similarity_hist),
                "template_score": float(similarity_template),
                "ssim_score": float(similarity_ssim)
            }
        }

        return result

    except Exception as e:
        logger.error(f"Error comparing faces: {str(e)}")
        return {"match": False, "similarity": 0.0, "error": str(e)}

def visualize_face_comparison(face1_path, face2_path, comparison_result):
    """
    Visualize face comparison results
    Args:
        face1_path: Path to document face image
        face2_path: Path to webcam face image
        comparison_result: Result from compare_faces function
    """
    try:
        # Load images
        face1 = cv2.imread(face1_path)
        face2 = cv2.imread(face2_path)

        if face1 is None or face2 is None:
            raise Exception("Failed to load face images")

        # Convert to RGB
        face1_rgb = cv2.cvtColor(face1, cv2.COLOR_BGR2RGB)
        face2_rgb = cv2.cvtColor(face2, cv2.COLOR_BGR2RGB)

        # Resize to same height
        height = 200
        width1 = int(face1_rgb.shape[1] * height / face1_rgb.shape[0])
        width2 = int(face2_rgb.shape[1] * height / face2_rgb.shape[0])

        face1_resized = cv2.resize(face1_rgb, (width1, height))
        face2_resized = cv2.resize(face2_rgb, (width2, height))

        # Create figure with two subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

        # Display images
        ax1.imshow(face1_resized)
        ax1.set_title("CIN Document Face")
        ax1.axis("off")

        ax2.imshow(face2_resized)
        ax2.set_title("Your Face Photo")
        ax2.axis("off")

        # Set figure title based on comparison result
        if comparison_result["match"]:
            match_text = "✅ MATCH"
            color = "green"
        else:
            match_text = "❌ NO MATCH"
            color = "red"

        similarity = comparison_result["similarity"]
        plt.suptitle(f"Face Comparison: {match_text} (Similarity: {similarity:.2f})",
                    color=color, fontsize=16)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        logger.error(f"Error visualizing face comparison: {str(e)}")
        print(f"Could not visualize comparison: {str(e)}")

def normalize_name(name):
    """
    Normalize a name for comparison with Arabic support
    Args:
        name: Input name string

    Returns:
        Normalized name
    """
    if not name:
        return ""

    # Handle Arabic text
    is_arabic = any('\u0600' <= c <= '\u06FF' for c in name)

    if is_arabic:
        # Arabic-specific normalization
        # Remove diacritics (harakat)
        normalized = ''.join(c for c in name if not '\u064B' <= c <= '\u0652')

        # Remove tatweel (elongation character)
        normalized = normalized.replace('\u0640', '')

        # Convert to lowercase and remove extra spaces
        normalized = normalized.lower()
        normalized = re.sub(r'\s+', ' ', normalized)

        return normalized.strip()
    else:
        # Latin text normalization (existing code)
        # Convert to lowercase
        name = name.lower()

        # Replace multiple spaces with single space
        name = re.sub(r'\s+', ' ', name)

        # Remove punctuation
        name = re.sub(r'[^\w\s]', '', name)

        # Standardize certain name elements
        replacements = {
            'mohamed': 'mohammed',
            'muhammed': 'mohammed',
            'mohamad': 'mohammed',
            'abd': 'abdul',
            'ben': 'bin'
        }

        words = name.split()
        normalized_words = []

        for word in words:
            if word in replacements:
                normalized_words.append(replacements[word])
            else:
                normalized_words.append(word)

        return ' '.join(normalized_words).strip()

def compare_names(name1, name2):
    """
    Compare two names and calculate similarity with Arabic support
    Args:
        name1: First name (e.g., from OCR)
        name2: Second name (e.g., from user input)

    Returns:
        Dictionary with similarity score and match result
    """
    if not name1 or not name2:
        return {"match": False, "similarity": 0.0}

    # Check if either name is in Arabic
    is_arabic1 = any('\u0600' <= c <= '\u06FF' for c in name1)
    is_arabic2 = any('\u0600' <= c <= '\u06FF' for c in name2)

    # If one is Arabic and the other isn't, likely dealing with transliteration
    # or they're completely different languages - need special handling
    if is_arabic1 != is_arabic2:
        # For mixed script comparison, we need to be more lenient
        # Look for partial substring matches after normalization
        norm1 = normalize_name(name1)
        norm2 = normalize_name(name2)

        # Check if shorter name is contained within longer one
        shorter = norm1 if len(norm1) < len(norm2) else norm2
        longer = norm2 if len(norm1) < len(norm2) else norm1

        if shorter in longer:
            similarity = len(shorter) / len(longer)
            return {"match": similarity > 0.5, "similarity": similarity}

        # If no direct containment, check for word overlap
        words1 = set(norm1.split())
        words2 = set(norm2.split())

        intersection = len(words1.intersection(words2))
        if intersection > 0:
            similarity = intersection / max(len(words1), len(words2))
            return {"match": similarity > 0.3, "similarity": similarity}

        return {"match": False, "similarity": 0.1}  # Small baseline similarity

    # Normal comparison for same script
    # Normalize both names
    norm1 = normalize_name(name1)
    norm2 = normalize_name(name2)

    # If exact match after normalization
    if norm1 == norm2:
        return {"match": True, "similarity": 1.0}

    # Split into words and compare overlap
    words1 = set(norm1.split())
    words2 = set(norm2.split())

    # For Arabic, we need to be more lenient
    threshold = 0.5 if is_arabic1 else 0.7

    # Check if all words in the shorter name are in the longer name
    if words1.issubset(words2) or words2.issubset(words1):
        # Calculate similarity based on word overlap
        common_words = words1.intersection(words2)
        total_words = max(len(words1), len(words2))

        if total_words > 0:
            similarity = len(common_words) / total_words
            return {"match": similarity > threshold, "similarity": similarity}

    # If no subset relationship, calculate token similarity using Jaccard index
    if not words1 or not words2:
        return {"match": False, "similarity": 0.0}

    intersection = len(words1.intersection(words2))
    union = len(words1.union(words2))

    similarity = intersection / union if union > 0 else 0.0
    return {"match": similarity > threshold, "similarity": similarity}

def verify_identity(cin_path, face_path, input_name):
    """
    Complete identity verification pipeline with improved Arabic support
    Args:
        cin_path: Path to CIN document image
        face_path: Path to face image (from webcam or upload)
        input_name: Name provided by the user

    Returns:
        Dictionary with verification results
    """
    result = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "ocr_data": {
            "extracted_name": None,
            "raw_text": ""
        },
        "name_verification": {
            "match": False,
            "similarity": 0.0
        },
        "face_detection": {
            "document_face_detected": False,
            "user_face_detected": True if face_path else False
        },
        "face_verification": {
            "match": False,
            "similarity": 0.0
        },
        "overall_verification": {
            "verified": False,
            "confidence": 0.0,
            "flags": []
        }
    }

    # Step 1: Load and process the CIN image
    cin_image = load_image(cin_path)
    if cin_image is None:
        result["overall_verification"]["flags"].append("Failed to load CIN image")
        return result

    processed_image, gray_image, thresh_image = preprocess_image(cin_image)

    # Step 2: Perform OCR to extract text from CIN - now with better Arabic support
    ocr_results = extract_text_with_easyocr(processed_image)
    ocr_text = " ".join([r[1] for r in ocr_results])
    result["ocr_data"]["raw_text"] = ocr_text

    # Log all detected text for debugging
    print(f"Detected text blocks: {len(ocr_results)}")
    for idx, (_, text, conf) in enumerate(ocr_results):
        print(f"  Text {idx}: {text} (Confidence: {conf:.2f})")

    # Step 3: Extract name from OCR results with improved Arabic handling
    extracted_name = extract_name_from_cin(ocr_results)
    result["ocr_data"]["extracted_name"] = extracted_name
    print(f"Extracted name: {extracted_name}")

    # Step 4: Verify name if extracted - with Arabic support
    if extracted_name:
        name_comparison = compare_names(extracted_name, input_name)
        result["name_verification"]["match"] = name_comparison["match"]
        result["name_verification"]["similarity"] = name_comparison["similarity"]
        print(f"Name comparison: {input_name} vs {extracted_name} = {name_comparison['similarity']:.2f}")

        if not name_comparison["match"]:
            result["overall_verification"]["flags"].append("Name mismatch between CIN and input")
    else:
        result["overall_verification"]["flags"].append("Could not extract name from CIN")

    # Step 5: Detect face in CIN document - improved parameters for Arabic documents
    document_faces = detect_faces_in_document(processed_image)
    result["face_detection"]["document_face_detected"] = len(document_faces) > 0
    print(f"Faces detected in document: {len(document_faces)}")

    doc_face_path = None
    if result["face_detection"]["document_face_detected"]:
        # Extract and save document face
        doc_face = extract_face_from_document(processed_image, document_faces)
        if doc_face is not None:
            doc_face_path = os.path.join('processed_results', f'cin_face_{datetime.now().strftime("%Y%m%d%H%M%S")}.jpg')
            cv2.imwrite(doc_face_path, cv2.cvtColor(doc_face, cv2.COLOR_RGB2BGR))
            print(f"Extracted face saved to: {doc_face_path}")
    else:
        result["overall_verification"]["flags"].append("No face detected in CIN document")

    # Step 6: Compare faces if both available - with more permissive threshold for verification
    if doc_face_path and face_path:
        face_comparison = compare_faces(doc_face_path, face_path)
        result["face_verification"]["match"] = face_comparison["match"]
        result["face_verification"]["similarity"] = face_comparison["similarity"]
        print(f"Face comparison similarity: {face_comparison['similarity']:.2f}")

        if not face_comparison["match"]:
            result["overall_verification"]["flags"].append("Face mismatch between CIN and user face image")

    # Step 7: Calculate overall verification result with more weight on name match for ID verification
    if result["name_verification"]["match"]:
        # If name matches perfectly, lower the face similarity threshold
        face_threshold = 0.35  # Lower threshold when name matches
    else:
        face_threshold = 0.5  # Higher threshold if name doesn't match

    # Calculate verification based on combination of face and name
    if result["face_verification"]["similarity"] >= face_threshold:
        # If face similarity passes threshold, verify
        if result["name_verification"]["match"]:
            # Both name and face match
            name_weight = 0.6
            face_weight = 0.4
        else:
            # Only face matches
            name_weight = 0.3
            face_weight = 0.7

        # Calculate weighted confidence
        overall_confidence = (name_weight * result["name_verification"]["similarity"] +
                            face_weight * result["face_verification"]["similarity"])

        # Verify if confidence is sufficient
        result["overall_verification"]["confidence"] = overall_confidence
        result["overall_verification"]["verified"] = overall_confidence > 0.5
    else:
        # Face similarity too low
        if result["name_verification"]["match"] and result["face_verification"]["similarity"] >= 0.2:
            # If name matches perfectly and there's at least some face similarity,
            # still verify but with lower confidence
            overall_confidence = 0.5 + (result["face_verification"]["similarity"] / 4)
            result["overall_verification"]["confidence"] = overall_confidence
            result["overall_verification"]["verified"] = True
        else:
            # Neither name nor face matches sufficiently
            result["overall_verification"]["verified"] = False
            result["overall_verification"]["confidence"] = 0

    # Debug information
    print(f"Verification result: {result['overall_verification']['verified']} with confidence {result['overall_verification']['confidence']:.2f}")

    return result

# Function to evaluate JavaScript from Python
def eval_js(js_code):
    """Execute JavaScript code and return the result"""
    from IPython.display import Javascript
    from google.colab import output
    return output.eval_js(js_code)

def create_direct_webcam_ui():
    """Create a webcam interface that captures and stores photos for direct verification"""

    # Generate unique IDs for HTML elements
    container_id = f"webcam-container-{uuid.uuid4().hex[:8]}"
    video_id = f"webcam-video-{uuid.uuid4().hex[:8]}"
    canvas_id = f"webcam-canvas-{uuid.uuid4().hex[:8]}"
    capture_btn_id = f"capture-btn-{uuid.uuid4().hex[:8]}"
    close_btn_id = f"close-btn-{uuid.uuid4().hex[:8]}"
    status_id = f"status-{uuid.uuid4().hex[:8]}"
    output_id = f"output-{uuid.uuid4().hex[:8]}"
    hidden_field_id = f"webcam-data-{uuid.uuid4().hex[:8]}"
    face_guide_id = f"face-guide-{uuid.uuid4().hex[:8]}"

    # Create a hidden input field where we'll store the webcam image data
    html = f"""
    <div id="{container_id}" style="width: 100%; max-width: 800px; margin: 0 auto; padding: 15px;
               background-color: #f8f9fa; border: 1px solid #ddd; border-radius: 5px;">
        <h3 style="text-align: center; margin-bottom: 15px;">Face Photo Capture</h3>

        <div style="text-align: center; position: relative;">
            <!-- Video element -->
            <div style="position: relative; display: inline-block;">
                <video id="{video_id}" width="400" height="300" autoplay playsinline
                       style="background-color: #000; border: 1px solid #444;"></video>

                <!-- Much smaller face alignment guide -->
                <div id="{face_guide_id}" style="
                    position: absolute;
                    top: 50%;
                    left: 50%;
                    transform: translate(-50%, -55%); /* Higher position for just the face */
                    width: 80px;  /* Much smaller width to focus only on face */
                    height: 80px; /* Square shape for just the face */
                    border: 2px dashed #FF4500; /* Bright orange color for better visibility */
                    border-radius: 100%; /* Circular shape for face */
                    box-shadow: 0 0 0 2000px rgba(0,0,0,0.15);
                    pointer-events: none;
                    z-index: 100;
                ">
                    <div style="
                        position: absolute;
                        bottom: -30px;
                        left: -60px;
                        right: -60px;
                        color: white;
                        text-shadow: 1px 1px 2px black;
                        font-size: 13px;
                    ">
                        Align ONLY your face in the circle
                    </div>
                </div>
            </div>

            <!-- Hidden canvas and data field for capturing -->
            <canvas id="{canvas_id}" style="display: none;"></canvas>
            <input type="hidden" id="{hidden_field_id}" value="">

            <!-- Controls -->
            <div style="margin-top: 10px;">
                <button id="{capture_btn_id}" disabled
                        style="padding: 8px 20px; background-color: #28a745; color: white;
                               border: none; border-radius: 4px; cursor: pointer;">
                    Capture Photo
                </button>
                <button id="{close_btn_id}"
                        style="padding: 8px 20px; background-color: #dc3545; color: white;
                               border: none; border-radius: 4px; cursor: pointer; margin-left: 10px; display: none;">
                    Close Camera
                </button>
            </div>

            <!-- Status area -->
            <div id="{status_id}" style="margin-top: 10px; min-height: 20px;">
                <p>Initializing camera...</p>
            </div>

            <!-- Output area for captured photo -->
            <div id="{output_id}" style="margin-top: 15px;"></div>
        </div>
    </div>
    """

    js = f"""
    // Direct webcam implementation
    (function() {{
        // Get DOM elements
        const video = document.getElementById('{video_id}');
        const canvas = document.getElementById('{canvas_id}');
        const captureBtn = document.getElementById('{capture_btn_id}');
        const closeBtn = document.getElementById('{close_btn_id}');
        const statusArea = document.getElementById('{status_id}');
        const outputArea = document.getElementById('{output_id}');
        const hiddenField = document.getElementById('{hidden_field_id}');
        const faceGuide = document.getElementById('{face_guide_id}');

        // Store stream reference globally so we can access it from other functions
        window.webcamStream = null;

        // Initialize webcam
        async function initCamera() {{
            statusArea.innerHTML = '<p>Requesting camera access...</p>';

            try {{
                // Request camera with higher resolution to improve face detection
                window.webcamStream = await navigator.mediaDevices.getUserMedia({{
                    video: {{
                        width: {{ ideal: 1280 }},
                        height: {{ ideal: 720 }},
                        facingMode: 'user'
                    }}
                }});

                // Set stream to video element
                video.srcObject = window.webcamStream;

                // Enable capture button when video starts playing
                video.onloadedmetadata = () => {{
                    video.play();
                    captureBtn.disabled = false;
                    statusArea.innerHTML = '<p style="color: green;">Camera ready! Position only your facial features inside the orange circle.</p>';

                    // Adjust face guide size based on actual video dimensions
                    const videoWidth = video.videoWidth || 400;

                    // Make the face guide approximately 15% of video width for just facial features
                    const faceGuideSize = Math.round(videoWidth * 0.15);

                    faceGuide.style.width = faceGuideSize + 'px';
                    faceGuide.style.height = faceGuideSize + 'px';
                }};

            }} catch (error) {{
                console.error('Camera error:', error);
                statusArea.innerHTML = `
                    <p style="color: red;">Camera access error: ${{error.message || error.name || 'Unknown error'}}</p>
                    <p>Please check your browser permissions and try again.</p>
                `;
            }}
        }}

        // Function to close camera stream
        function closeCamera() {{
            if (window.webcamStream) {{
                // Stop all tracks
                window.webcamStream.getTracks().forEach(track => {{
                    track.stop();
                }});

                // Clear video source
                video.srcObject = null;
                window.webcamStream = null;

                // Update UI
                video.style.backgroundColor = "#333";
                captureBtn.disabled = true;
                closeBtn.style.display = 'none';

                statusArea.innerHTML = '<p style="color: blue;">✓ Camera closed successfully</p>';
            }}
        }}

        // Make closeCamera available globally
        window.closeWebcam = closeCamera;

        // Custom function to mark facial area for processing
        function processWithFacialRegion() {{
            // Set canvas size to match video
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            const ctx = canvas.getContext('2d');

            // Draw video frame to canvas
            ctx.drawImage(video, 0, 0);

            // Get face guide dimensions and position
            const guideWidth = faceGuide.offsetWidth;
            const guideHeight = faceGuide.offsetHeight;

            // Calculate face guide position in video coordinates
            const videoRect = video.getBoundingClientRect();
            const guideRect = faceGuide.getBoundingClientRect();

            const centerX = (guideRect.left + guideRect.width/2 - videoRect.left) * (video.videoWidth / videoRect.width);
            const centerY = (guideRect.top + guideRect.height/2 - videoRect.top) * (video.videoHeight / videoRect.height);
            const radiusX = (guideWidth/2) * (video.videoWidth / videoRect.width);
            const radiusY = (guideHeight/2) * (video.videoHeight / videoRect.height);

            // Draw a subtle indicator on the captured image for debugging (won't be visible in normal use)
            ctx.strokeStyle = 'rgba(255, 69, 0, 0.05)';  // Almost invisible orange
            ctx.lineWidth = 1;
            ctx.beginPath();
            ctx.ellipse(centerX, centerY, radiusX, radiusY, 0, 0, 2 * Math.PI);
            ctx.stroke();

            return canvas.toDataURL('image/jpeg', 0.95);
        }}

        // Capture button click handler
        captureBtn.addEventListener('click', () => {{
            try {{
                // Use our custom function to process the image with face focus
                const imageData = processWithFacialRegion();

                // Store the image data in the hidden field for Python to access
                hiddenField.value = imageData;

                // Show captured image
                outputArea.innerHTML = `
                    <div style="margin-top: 15px;">
                        <img src="${{imageData}}" style="max-width: 320px; border: 2px solid green;">
                        <p style="color: green; margin-top: 10px;">✅ Photo captured successfully!</p>
                        <p style="color: #007bff; margin-top: 5px;">Click "VERIFY IDENTITY" to proceed with verification</p>
                    </div>
                `;

                statusArea.innerHTML = '<p style="color: green;">✅ Photo captured! Click "VERIFY IDENTITY" to proceed.</p>';

                // Set a global variable that Python can check
                window.webcamPhotoCaptured = true;

                // Show close camera button
                closeBtn.style.display = 'inline-block';

            }} catch (error) {{
                console.error('Error capturing photo:', error);
                outputArea.innerHTML = `
                    <p style="color: red;">Error capturing photo: ${{error.message}}</p>
                `;
            }}
        }});

        // Close button click handler
        closeBtn.addEventListener('click', closeCamera);

        // Start camera
        initCamera();

        // Clean up when the element is removed from the DOM
        const observer = new MutationObserver((mutations) => {{
            mutations.forEach((mutation) => {{
                mutation.removedNodes.forEach((node) => {{
                    if (node === video || node.contains(video)) {{
                        closeCamera();
                        observer.disconnect();
                    }}
                }});
            }});
        }});

        observer.observe(document.body, {{ childList: true, subtree: true }});
    }})();
    """

    # Display the webcam interface
    display(HTML(html))
    display(Javascript(js))

    return hidden_field_id
def get_webcam_data(hidden_field_id):
    """Get webcam data from the hidden field using JavaScript"""
    from IPython.display import Javascript
    from IPython.core.display import display

    # JavaScript to retrieve the value
    js_code = f"""
    var dataField = document.getElementById('{hidden_field_id}');
    var imageData = dataField ? dataField.value : '';
    imageData;
    """

    # Execute JavaScript and get the result
    result = eval_js(js_code)
    return result

def create_streamlined_verification_ui():
    """Create a streamlined verification interface with direct webcam integration"""

    # Title and instructions
    title = widgets.HTML(value="<h1>🆔 CIN Document Verification System</h1>")
    instructions = widgets.HTML(
        value="<p>This system verifies identity by comparing a CIN (ID card) with a face photo and user information.</p>"
        "<p>Please complete all steps below for verification:</p>"
    )

    # 1. Name input
    name_label = widgets.HTML(value="<h3>1. Enter Your Full Name</h3>")
    name_input = widgets.Text(
        description="Full Name:",
        placeholder="As shown on your CIN",
        layout=widgets.Layout(width='60%')
    )

    # 2. CIN upload
    cin_label = widgets.HTML(value="<h3>2. Upload Your CIN Document</h3>")
    cin_upload = FileUpload(
        accept='image/*',
        multiple=False,
        description="CIN Image:",
        button_style="primary",
        layout=widgets.Layout(width='auto')
    )

    # 3. Face Photo capture with webcam
    face_label = widgets.HTML(value="<h3>3. Capture Your Face Photo</h3>")

    webcam_instructions = widgets.HTML(value="""
    <div style="background-color: #e9ecef; padding: 10px; border-radius: 5px; margin-bottom: 10px;">
        <p><strong>Webcam Instructions:</strong></p>
        <ol>
            <li>Allow camera access when prompted</li>
            <li>Position your face in the center of the frame</li>
            <li>Click "Capture Photo" to take your picture</li>
            <li>Click "VERIFY IDENTITY" to proceed with verification</li>
        </ol>
    </div>
    """)

    # Add webcam UI
    webcam_output = widgets.Output()
    webcam_field_id = None

    with webcam_output:
        webcam_field_id = create_direct_webcam_ui()

    face_section = widgets.VBox([
        webcam_instructions,
        webcam_output
    ])

    # 4. Verify button
    verify_button = Button(
        description="VERIFY IDENTITY",
        button_style="success",
        icon="check",
        layout=widgets.Layout(width="200px", margin="20px 0px")
    )

    # Output areas
    processing_output = Output()
    results_output = Output()

    # Function to display verification results
    def display_verification_results(result, cin_path, face_path):
        """Display verification results in a structured way"""
        with results_output:
            results_output.clear_output()

            # Header style
            display(widgets.HTML(value="""
            <style>
                .result-header { font-weight: bold; margin-top: 15px; color: #333; }
                .result-section { padding: 12px; border: 1px solid #ddd; border-radius: 5px; margin: 15px 0; background-color: #f9f9f9; }
                .success-text { color: green; font-weight: bold; }
                .error-text { color: #d9534f; font-weight: bold; }
            </style>
            """))

            # Load images for display
            cin_img = load_image(cin_path)
            face_img = load_image(face_path)

            # Display both input images side by side
            if cin_img is not None and face_img is not None:
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
                ax1.imshow(cin_img)
                ax1.set_title("CIN Document")
                ax1.axis('off')

                ax2.imshow(face_img)
                ax2.set_title("Face Photo")
                ax2.axis('off')

                plt.tight_layout()
                plt.show()

            # Overall verification result
            verification_status = "✅ VERIFIED" if result["overall_verification"]["verified"] else "❌ NOT VERIFIED"
            confidence = result["overall_verification"]["confidence"]

            display(widgets.HTML(value=f"""
                <div class='result-section'>
                    <h2 style='color: {'green' if result["overall_verification"]["verified"] else "red"}; text-align: center;'>
                        {verification_status} (Confidence: {confidence:.2f})
                    </h2>
                </div>
            """))

            # Show OCR and name verification in one section
            display(widgets.HTML(value="""<div class='result-section'>"""))

            # OCR Data
            display(widgets.HTML(value="<p class='result-header'>📄 OCR Data:</p>"))
            extracted_name = result["ocr_data"]["extracted_name"] or "Not found"
            display(widgets.HTML(value=f"<p>Extracted Name: <b>{extracted_name}</b></p>"))

            # Name verification
            display(widgets.HTML(value="<p class='result-header'>👤 Name Verification:</p>"))
            name_match = "✅ Match" if result["name_verification"]["match"] else "❌ No match"
            name_class = "success-text" if result["name_verification"]["match"] else "error-text"
            display(widgets.HTML(value=f"""
                <p>Input Name: <b>{name_input.value}</b></p>
                <p>Name Comparison: <span class='{name_class}'>{name_match}</span>
                   (Similarity: {result['name_verification']['similarity']:.2f})</p>
            """))

            display(widgets.HTML(value="""</div>"""))

            # Show OCR results visualization if available
            if cin_img is not None:
                ocr_results = extract_text_with_easyocr(cin_img)
                ocr_visualization = visualize_ocr_results(cin_img, ocr_results)
                if ocr_visualization is not None:
                    plt.figure(figsize=(8, 5))
                    plt.imshow(ocr_visualization)
                    plt.title("OCR Text Detection")
                    plt.axis('off')
                    plt.show()

            # Face verification section
            display(widgets.HTML(value="""<div class='result-section'>"""))
            display(widgets.HTML(value="<p class='result-header'>🔍 Face Verification:</p>"))

            # Face detection in document
            doc_face = "✅ Detected" if result["face_detection"]["document_face_detected"] else "❌ Not detected"
            doc_face_class = "success-text" if result["face_detection"]["document_face_detected"] else "error-text"
            display(widgets.HTML(value=f"<p>Face in CIN: <span class='{doc_face_class}'>{doc_face}</span></p>"))

            # If face detection was successful, show visualization
            if cin_img is not None and result["face_detection"]["document_face_detected"]:
                document_faces = detect_faces_in_document(cin_img)
                if document_faces:
                    faces_visualization = visualize_faces(cin_img, document_faces)
                    plt.figure(figsize=(8, 5))
                    plt.imshow(faces_visualization)
                    plt.title("Face Detection in CIN")
                    plt.axis('off')
                    plt.show()

            # If face comparison was done, display results
            if result["face_detection"]["document_face_detected"] and result["face_detection"]["user_face_detected"]:
                face_match = "✅ Match" if result["face_verification"]["match"] else "❌ No match"
                face_class = "success-text" if result["face_verification"]["match"] else "error-text"
                similarity = result["face_verification"]["similarity"]

                display(widgets.HTML(value=f"""
                    <p>Face Comparison: <span class='{face_class}'>{face_match}</span>
                       (Similarity: {similarity:.2f})</p>
                """))

                # Show face comparison visualization if paths are available
                if cin_path and face_path:
                    # Find document face and extract to temp file for comparison
                    cin_img = load_image(cin_path)
                    document_faces = detect_faces_in_document(cin_img)
                    if document_faces:
                        doc_face = extract_face_from_document(cin_img, document_faces)
                        if doc_face is not None:
                            doc_face_path = os.path.join('processed_results', 'temp_doc_face.jpg')
                            cv2.imwrite(doc_face_path, cv2.cvtColor(doc_face, cv2.COLOR_RGB2BGR))

                            # Visualize face comparison
                            face_comparison = compare_faces(doc_face_path, face_path)
                            visualize_face_comparison(doc_face_path, face_path, face_comparison)

            display(widgets.HTML(value="""</div>"""))

            # Show any verification flags
            if result["overall_verification"]["flags"]:
                display(widgets.HTML(value="""<div class='result-section'>"""))
                display(widgets.HTML(value="<p class='result-header'>⚠️ Verification Flags:</p>"))
                flags_html = "<ul>"
                for flag in result["overall_verification"]["flags"]:
                    flags_html += f"<li>{flag}</li>"
                flags_html += "</ul>"
                display(widgets.HTML(value=flags_html))
                display(widgets.HTML(value="""</div>"""))

            # Summary
            if result["overall_verification"]["verified"]:
                display(widgets.HTML(value="""
                    <div class='result-section' style='background-color: #dff0d8; border-color: #d6e9c6;'>
                        <h3 style='color: green; text-align: center;'>Identity verification successful! ✅</h3>
                    </div>
                """))
            else:
                display(widgets.HTML(value="""
                    <div class='result-section' style='background-color: #f2dede; border-color: #ebccd1;'>
                        <h3 style='color: #a94442; text-align: center;'>Identity verification failed! ❌</h3>
                    </div>
                """))

    # Handle verification button click
    def on_verify_clicked(b):
        with processing_output:
            processing_output.clear_output()
            print("⏳ Starting verification process...")

            # 1. Validate name input
            if not name_input.value:
                print("❌ Please enter your full name")
                return

            # 2. Validate CIN upload
            cin_path = None
            if cin_upload.value:
                cin_path = save_uploaded_file(cin_upload)
                if cin_path:
                    print(f"✅ CIN document saved: {cin_path}")
                else:
                    print("❌ Failed to save CIN document")
                    return
            else:
                print("❌ Please upload your CIN document")
                return

            # 3. Get webcam photo data
            webcam_data = get_webcam_data(webcam_field_id)

            if not webcam_data:
                print("❌ No webcam photo found. Please capture a photo using the webcam.")
                return

            # Save the webcam data as an image file
            face_path = save_base64_image(webcam_data)

            if face_path:
                print(f"✅ Webcam photo saved: {face_path}")
            else:
                print("❌ Failed to save webcam photo")
                return

            # 4. Run verification
            print("\n🔍 Running identity verification...")
            verification_result = verify_identity(cin_path, face_path, name_input.value)

            # 5. Display results
            display_verification_results(verification_result, cin_path, face_path)

            # 6. Close the webcam after verification is complete
            display(Javascript("if (window.closeWebcam) { window.closeWebcam(); }"))

    # Connect button to handler
    verify_button.on_click(on_verify_clicked)

    # Assemble UI
    ui = VBox([
        title,
        instructions,

        # Main verification flow
        name_label,
        name_input,

        cin_label,
        cin_upload,

        face_label,
        face_section,

        verify_button,
        processing_output,

        # Results section
        widgets.HTML(value="<hr style='margin: 30px 0; border-top: 1px solid #ddd;'>"),
        widgets.HTML(value="<h2>Verification Results:</h2>"),
        results_output
    ])

    return ui

# Create and display the UI
verification_ui = create_streamlined_verification_ui()
display(verification_ui)

Starting EasyOCR initialization...
Initializing EasyOCR with English only...


✅ EasyOCR initialized successfully with English
Adding French language model...


✅ French language model added successfully
Adding Arabic language model...
✅ Arabic language model added successfully
